# Tariff Reference Profiling

Profiles and validates the dynamic Time-of-Use tariff schedule supplied alongside the London Smart Meter dataset.

## Purpose

- Inspect the tariff reference workbook
- Validate the available worksheet and source schema
- Confirm tariff schedule coverage
- Validate half-hour interval consistency
- Profile High, Normal and Low tariff bands
- Check timestamp uniqueness and completeness
- Prepare the reference schedule for Gold-layer enrichment

> The tariff reference describes dynamic tariff bands for 2013. It is separate from the household `TariffType` classification (`Std` / `ToU`) contained in the smart-meter readings.

In [1]:
from notebookutils import mssparkutils

tariff_path = "Files/landing/tariffs/"

files = mssparkutils.fs.ls(tariff_path)

for file in files:
    print(
        file.name,
        file.size,
        file.path
    )

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 3, Finished, Available, Finished, False)

data_fb0685ec-9f7b-413f-a140-2e0ef3e6719e_1b0dffb4-1fd4-4531-8858-f6816eec439f.xlsx 245384 abfss://0b809d41-0d46-4aa4-b032-f61ff11a691f@onelake.dfs.fabric.microsoft.com/e1c5d971-3a46-4610-899b-91f5d706864e/Files/landing/tariffs/data_fb0685ec-9f7b-413f-a140-2e0ef3e6719e_1b0dffb4-1fd4-4531-8858-f6816eec439f.xlsx


## 1. Load Tariff Reference Workbook

Load the tariff schedule from the OneLake landing area for profiling and validation.

In [3]:
local_tariff_path = "/tmp/Tariffs.xlsx"

mssparkutils.fs.cp(
    "Files/landing/tariffs/data_fb0685ec-9f7b-413f-a140-2e0ef3e6719e_1b0dffb4-1fd4-4531-8858-f6816eec439f.xlsx",
    f"file:{local_tariff_path}"
)

print("Tariff workbook copied locally.")

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 5, Finished, Available, Finished, False)

Tariff workbook copied locally.


## 2. Inspect Workbook Structure

Inspect the workbook structure and identify the worksheet containing the usable tariff schedule.

In [4]:
import pandas as pd

excel_file = pd.ExcelFile(local_tariff_path)

print("WORKBOOK SHEETS")
print("-" * 50)

for sheet in excel_file.sheet_names:
    print(sheet)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 6, Finished, Available, Finished, False)

WORKBOOK SHEETS
--------------------------------------------------
Sheet1
Sheet2
Sheet3


## 3. Standardise Tariff Reference Fields

Standardise the tariff timestamp and band fields into a consistent analytical structure for validation and downstream enrichment.

In [5]:
for sheet in excel_file.sheet_names:

    print("\n")
    print("=" * 70)
    print(f"SHEET: {sheet}")
    print("=" * 70)

    sheet_df = pd.read_excel(
        local_tariff_path,
        sheet_name=sheet
    )

    print(f"Rows: {len(sheet_df):,}")
    print(f"Columns: {len(sheet_df.columns)}")

    print("\nColumns:")
    print(sheet_df.columns.tolist())

    print("\nFirst 10 rows:")
    print(sheet_df.head(10).to_string(index=False))

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 8, Finished, Available, Finished, False)



SHEET: Sheet1
Rows: 17,520
Columns: 2

Columns:
['TariffDateTime', 'Tariff']

First 10 rows:
     TariffDateTime Tariff
2013-01-01 00:00:00 Normal
2013-01-01 00:30:00 Normal
2013-01-01 01:00:00 Normal
2013-01-01 01:30:00 Normal
2013-01-01 02:00:00 Normal
2013-01-01 02:30:00 Normal
2013-01-01 03:00:00 Normal
2013-01-01 03:30:00 Normal
2013-01-01 04:00:00 Normal
2013-01-01 04:30:00 Normal


SHEET: Sheet2
Rows: 0
Columns: 0

Columns:
[]

First 10 rows:
Empty DataFrame
Columns: []
Index: []


SHEET: Sheet3
Rows: 0
Columns: 0

Columns:
[]

First 10 rows:
Empty DataFrame
Columns: []
Index: []


In [6]:
for sheet in excel_file.sheet_names:

    sheet_df = pd.read_excel(
        local_tariff_path,
        sheet_name=sheet
    )

    print("\n")
    print("=" * 70)
    print(f"PROFILE: {sheet}")
    print("=" * 70)

    print("\nData types:")
    print(sheet_df.dtypes)

    print("\nNull counts:")
    print(sheet_df.isnull().sum())

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 10, Finished, Available, Finished, False)



PROFILE: Sheet1

Data types:
TariffDateTime    datetime64[ns]
Tariff                    object
dtype: object

Null counts:
TariffDateTime    0
Tariff            0
dtype: int64


PROFILE: Sheet2

Data types:
Series([], dtype: object)

Null counts:
Series([], dtype: float64)


PROFILE: Sheet3

Data types:
Series([], dtype: object)

Null counts:
Series([], dtype: float64)


In [7]:
tariff_pd = pd.read_excel(
    local_tariff_path,
    sheet_name="Sheet1"
)

print(tariff_pd.head())
print(tariff_pd.dtypes)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 12, Finished, Available, Finished, False)

       TariffDateTime  Tariff
0 2013-01-01 00:00:00  Normal
1 2013-01-01 00:30:00  Normal
2 2013-01-01 01:00:00  Normal
3 2013-01-01 01:30:00  Normal
4 2013-01-01 02:00:00  Normal
TariffDateTime    datetime64[ns]
Tariff                    object
dtype: object


## 4. Profile Tariff Schedule Coverage

Validate the size and temporal coverage of the dynamic tariff schedule.

In [8]:
print("TARIFF SCHEDULE PROFILE")
print("-" * 50)

print(f"Rows: {len(tariff_pd):,}")
print(f"Min datetime: {tariff_pd['TariffDateTime'].min()}")
print(f"Max datetime: {tariff_pd['TariffDateTime'].max()}")
print(
    f"Distinct timestamps: "
    f"{tariff_pd['TariffDateTime'].nunique():,}"
)

print(
    f"Duplicate timestamps: "
    f"{tariff_pd['TariffDateTime'].duplicated().sum():,}"
)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 14, Finished, Available, Finished, False)

TARIFF SCHEDULE PROFILE
--------------------------------------------------
Rows: 17,520
Min datetime: 2013-01-01 00:00:00
Max datetime: 2013-12-31 23:30:00
Distinct timestamps: 17,520
Duplicate timestamps: 0


### Schedule Completeness

The reference contains exactly **17,520 unique half-hour periods**:

`365 days × 48 half-hour slots = 17,520`

This confirms complete half-hour schedule coverage for calendar year 2013.

## 5. Profile Dynamic Tariff Bands

Profile the number of half-hour periods assigned to each dynamic tariff band.

In [9]:
print("TARIFF BAND DISTRIBUTION")
print("-" * 50)

print(
    tariff_pd["Tariff"]
    .value_counts(dropna=False)
)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 15, Finished, Available, Finished, False)

TARIFF BAND DISTRIBUTION
--------------------------------------------------
Tariff
Normal    15072
Low        1660
High        788
Name: count, dtype: int64


## 6. Validate Half-Hour Intervals

Confirm that consecutive tariff schedule timestamps follow the expected 30-minute interval.

In [10]:
tariff_sorted = (
    tariff_pd
    .sort_values("TariffDateTime")
    .copy()
)

tariff_sorted["IntervalMinutes"] = (
    tariff_sorted["TariffDateTime"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

print("INTERVAL DISTRIBUTION")
print("-" * 50)

print(
    tariff_sorted["IntervalMinutes"]
    .value_counts()
    .sort_index()
)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 16, Finished, Available, Finished, False)

INTERVAL DISTRIBUTION
--------------------------------------------------
IntervalMinutes
30.0    17519
Name: count, dtype: int64


## 7. Analyse Tariff Band Timing

Inspect when High, Normal and Low tariff periods occur throughout the day.

The schedule is treated as a dynamic reference rather than assuming fixed clock-time tariff bands.

In [11]:
tariff_pd["Hour"] = tariff_pd["TariffDateTime"].dt.hour
tariff_pd["Minute"] = tariff_pd["TariffDateTime"].dt.minute

band_time_profile = (
    tariff_pd
    .groupby(
        ["Tariff", "Hour", "Minute"]
    )
    .size()
    .reset_index(name="Occurrences")
    .sort_values(
        ["Tariff", "Hour", "Minute"]
    )
)

print(band_time_profile.to_string(index=False))

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 17, Finished, Available, Finished, False)

Tariff  Hour  Minute  Occurrences
  High     0       0            9
  High     0      30            9
  High     1       0            9
  High     1      30            9
  High     2       0            9
  High     2      30            9
  High     3       0            9
  High     3      30            9
  High     4       0            9
  High     4      30            9
  High     5       0            9
  High     5      30            9
  High     6       0            9
  High     6      30            9
  High     7       0           11
  High     7      30           11
  High     8       0           11
  High     8      30           11
  High     9       0           11
  High     9      30           11
  High    10       0           10
  High    10      30           10
  High    11       0           13
  High    11      30           13
  High    12       0           13
  High    12      30           13
  High    13       0           13
  High    13      30           13
  High    14  

In [12]:
tariff_source_pd = tariff_pd[
    ["TariffDateTime", "Tariff"]
].copy()

tariff_spark = spark.createDataFrame(tariff_source_pd)

tariff_spark.printSchema()
tariff_spark.show(10, truncate=False)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 18, Finished, Available, Finished, False)

root
 |-- TariffDateTime: timestamp (nullable = true)
 |-- Tariff: string (nullable = true)

+-------------------+------+
|TariffDateTime     |Tariff|
+-------------------+------+
|2013-01-01 00:00:00|Normal|
|2013-01-01 00:30:00|Normal|
|2013-01-01 01:00:00|Normal|
|2013-01-01 01:30:00|Normal|
|2013-01-01 02:00:00|Normal|
|2013-01-01 02:30:00|Normal|
|2013-01-01 03:00:00|Normal|
|2013-01-01 03:30:00|Normal|
|2013-01-01 04:00:00|Normal|
|2013-01-01 04:30:00|Normal|
+-------------------+------+
only showing top 10 rows



In [13]:
from pyspark.sql import functions as F

tariff_schedule = (
    tariff_spark

    .withColumnRenamed(
        "Tariff",
        "TariffBand"
    )

    .withColumn(
        "DateKey",
        F.date_format(
            F.col("TariffDateTime"),
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "TimeKey",
        (
            F.hour("TariffDateTime") * 100
            + F.minute("TariffDateTime")
        ).cast("int")
    )

    .select(
        "TariffDateTime",
        "DateKey",
        "TimeKey",
        "TariffBand"
    )
)

tariff_schedule.printSchema()
tariff_schedule.show(10, truncate=False)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 19, Finished, Available, Finished, False)

root
 |-- TariffDateTime: timestamp (nullable = true)
 |-- DateKey: integer (nullable = true)
 |-- TimeKey: integer (nullable = true)
 |-- TariffBand: string (nullable = true)

+-------------------+--------+-------+----------+
|TariffDateTime     |DateKey |TimeKey|TariffBand|
+-------------------+--------+-------+----------+
|2013-01-01 00:00:00|20130101|0      |Normal    |
|2013-01-01 00:30:00|20130101|30     |Normal    |
|2013-01-01 01:00:00|20130101|100    |Normal    |
|2013-01-01 01:30:00|20130101|130    |Normal    |
|2013-01-01 02:00:00|20130101|200    |Normal    |
|2013-01-01 02:30:00|20130101|230    |Normal    |
|2013-01-01 03:00:00|20130101|300    |Normal    |
|2013-01-01 03:30:00|20130101|330    |Normal    |
|2013-01-01 04:00:00|20130101|400    |Normal    |
|2013-01-01 04:30:00|20130101|430    |Normal    |
+-------------------+--------+-------+----------+
only showing top 10 rows



In [14]:
dim_date = spark.table("gold.dim_date")
dim_time = spark.table("gold.dim_time")

missing_dates = (
    tariff_schedule
    .join(
        dim_date.select("DateKey"),
        "DateKey",
        "left_anti"
    )
    .count()
)

missing_times = (
    tariff_schedule
    .join(
        dim_time.select("TimeKey"),
        "TimeKey",
        "left_anti"
    )
    .count()
)

duplicate_schedule_keys = (
    tariff_schedule
    .groupBy("DateKey", "TimeKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("TARIFF SCHEDULE VALIDATION")
print("-" * 50)
print(f"Rows:                    {tariff_schedule.count():,}")
print(f"Missing DateKeys:        {missing_dates:,}")
print(f"Missing TimeKeys:        {missing_times:,}")
print(f"Duplicate schedule keys: {duplicate_schedule_keys:,}")

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 20, Finished, Available, Finished, False)

TARIFF SCHEDULE VALIDATION
--------------------------------------------------
Rows:                    17,520
Missing DateKeys:        0
Missing TimeKeys:        0
Duplicate schedule keys: 0


In [15]:
tariff_schedule.groupBy(
    "TariffBand"
).count().orderBy(
    F.desc("count")
).show()

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 21, Finished, Available, Finished, False)

+----------+-----+
|TariffBand|count|
+----------+-----+
|    Normal|15072|
|       Low| 1660|
|      High|  788|
+----------+-----+



## 8. Build Gold Tariff Schedule

Transform the validated reference schedule into Gold-compatible Date and Time keys for downstream demand-pattern enrichment.

### Grain

**One row per Date × Half-Hour Time Slot**

The resulting reference contains:

- `TariffDateTime`
- `DateKey`
- `TimeKey`
- `TariffBand`

In [16]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS gold
""")

(
    tariff_schedule
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.tariff_schedule")
)

print("gold.tariff_schedule created successfully.")

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 22, Finished, Available, Finished, False)

gold.tariff_schedule created successfully.


## 9. Persist Tariff Schedule

Persist the validated reference as:

`gold.tariff_schedule`

This table acts as an enrichment reference for the Demand Pattern fact and is not exposed directly in the semantic model.

In [17]:
persisted_tariff = spark.table(
    "gold.tariff_schedule"
)

print(
    f"Persisted rows: "
    f"{persisted_tariff.count():,}"
)

persisted_tariff.groupBy(
    "TariffBand"
).count().show()

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 23, Finished, Available, Finished, False)

Persisted rows: 17,520
+----------+-----+
|TariffBand|count|
+----------+-----+
|      High|  788|
|       Low| 1660|
|    Normal|15072|
+----------+-----+



In [18]:
demand_df = spark.table("gold.fact_demand_pattern")
tariff_dim = spark.table("gold.dim_tariff")
schedule_df = spark.table("gold.tariff_schedule")

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 24, Finished, Available, Finished, False)

## 10. Validate Gold Reference Integrity

Confirm that every tariff schedule period resolves to the Gold Date and Time dimensions and that the Date × Time grain remains unique.

In [19]:
tou_demand = (
    demand_df.alias("f")

    .join(
        tariff_dim.alias("t"),
        F.col("f.TariffKey") == F.col("t.TariffKey"),
        "inner"
    )

    .filter(
        F.col("t.TariffType") == "ToU"
    )

    .join(
        schedule_df.alias("s"),
        (
            (F.col("f.DateKey") == F.col("s.DateKey"))
            &
            (F.col("f.TimeKey") == F.col("s.TimeKey"))
        ),
        "inner"
    )

    .select(
        F.col("f.DateKey"),
        F.col("f.TimeKey"),
        F.col("s.TariffBand"),
        F.col("f.TotalConsumptionKWh"),
        F.col("f.ReadingCount"),
        F.col("f.DistinctHouseholds")
    )
)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 25, Finished, Available, Finished, False)

In [20]:
tou_demand.agg(
    F.count("*").alias("Rows"),
    F.min("DateKey").alias("MinDateKey"),
    F.max("DateKey").alias("MaxDateKey")
).show()

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 26, Finished, Available, Finished, False)

+-----+----------+----------+
| Rows|MinDateKey|MaxDateKey|
+-----+----------+----------+
|17520|  20130101|  20131231|
+-----+----------+----------+



In [21]:
band_analysis = (
    tou_demand
    .groupBy("TariffBand")
    .agg(
        F.count("*").alias("DemandIntervals"),

        F.sum("ReadingCount")
        .alias("ReadingCount"),

        F.sum("TotalConsumptionKWh")
        .alias("TotalConsumptionKWh"),

        (
            F.sum("TotalConsumptionKWh")
            /
            F.sum("ReadingCount")
        ).alias("AvgConsumptionPerReadingKWh")
    )
    .orderBy("TariffBand")
)

band_analysis.show(
    truncate=False
)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 27, Finished, Available, Finished, False)

+----------+---------------+------------+-------------------+---------------------------+
|TariffBand|DemandIntervals|ReadingCount|TotalConsumptionKWh|AvgConsumptionPerReadingKWh|
+----------+---------------+------------+-------------------+---------------------------+
|High      |788            |849851      |191732.4079977     |0.22560708641597174        |
|Low       |1660           |1787739     |373580.2129888     |0.2089679830158653         |
|Normal    |15072          |16241847    |3149731.505933102  |0.1939269287497353         |
+----------+---------------+------------+-------------------+---------------------------+



In [22]:
band_values = {
    row["TariffBand"]:
        row["AvgConsumptionPerReadingKWh"]
    for row in band_analysis.collect()
}

for band, value in band_values.items():
    print(
        f"{band}: "
        f"{value:.6f} kWh per half-hour reading"
    )

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 28, Finished, Available, Finished, False)

High: 0.225607 kWh per half-hour reading
Low: 0.208968 kWh per half-hour reading
Normal: 0.193927 kWh per half-hour reading


In [23]:
if all(
    band in band_values
    for band in ["Low", "Normal", "High"]
):
    low = band_values["Low"]
    normal = band_values["Normal"]
    high = band_values["High"]

    print("\nRELATIVE DEMAND COMPARISON")
    print("-" * 50)

    print(
        f"High vs Normal: "
        f"{((high / normal) - 1) * 100:.2f}%"
    )

    print(
        f"High vs Low: "
        f"{((high / low) - 1) * 100:.2f}%"
    )

    print(
        f"Low vs Normal: "
        f"{((low / normal) - 1) * 100:.2f}%"
    )

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 29, Finished, Available, Finished, False)


RELATIVE DEMAND COMPARISON
--------------------------------------------------
High vs Normal: 16.34%
High vs Low: 7.96%
Low vs Normal: 7.76%


In [24]:
all_tariff_demand = (
    demand_df.alias("f")

    .join(
        tariff_dim.alias("t"),
        F.col("f.TariffKey") == F.col("t.TariffKey"),
        "inner"
    )

    .join(
        schedule_df.alias("s"),
        (
            (F.col("f.DateKey") == F.col("s.DateKey"))
            &
            (F.col("f.TimeKey") == F.col("s.TimeKey"))
        ),
        "inner"
    )

    .groupBy(
        F.col("t.TariffType").alias("TariffType"),
        F.col("s.TariffBand").alias("TariffBand")
    )

    .agg(
        F.sum("f.ReadingCount").alias("ReadingCount"),

        F.sum("f.TotalConsumptionKWh")
        .alias("TotalConsumptionKWh"),

        (
            F.sum("f.TotalConsumptionKWh")
            /
            F.sum("f.ReadingCount")
        ).alias("AvgConsumptionPerReadingKWh")
    )

    .orderBy(
        "TariffBand",
        "TariffType"
    )
)

all_tariff_demand.show(
    truncate=False
)

StatementMeta(, ea2abcdf-2756-4e30-8fa1-70a7be4b9f4f, 30, Finished, Available, Finished, False)

+----------+----------+------------+--------------------+---------------------------+
|TariffType|TariffBand|ReadingCount|TotalConsumptionKWh |AvgConsumptionPerReadingKWh|
+----------+----------+------------+--------------------+---------------------------+
|Std       |High      |3335609     |860402.2289926998   |0.25794456993991194        |
|ToU       |High      |849851      |191732.4079977      |0.22560708641597174        |
|Std       |Low       |7016376     |1517625.2559844998  |0.21629759522358832        |
|ToU       |Low       |1787739     |373580.2129888      |0.2089679830158653         |
|Std       |Normal    |63792795    |1.3522667001689307E7|0.21197796713075995        |
|ToU       |Normal    |16241847    |3149731.505933102   |0.1939269287497353         |
+----------+----------+------------+--------------------+---------------------------+



## Tariff Reference Summary

The tariff reference provides a complete and validated dynamic pricing schedule for 2013.

- **17,520** half-hour periods
- **365** days of complete schedule coverage
- **48** periods per day
- **15,072** Normal periods
- **1,660** Low periods
- **788** High periods
- **0** duplicate timestamps
- **0** missing Gold Date or Time keys

The validated schedule is persisted as `gold.tariff_schedule` and used to enrich the Demand Pattern fact with dynamic tariff-band context.

The source workbook defines tariff-band timing but does not contain tariff prices; therefore pricing values are not introduced into the Gold model.